# ConceptNet_lite para búsquedas de relaciones y recuperación de caminos

El propósito de esta notebook es experimentar las formas de búsqueda en el grafo de ConceptNet en ambas direcciones para cada palabra de EmoPro y encontrar una forma de recuperar el path de la búsqueda. 

In [2]:
# Importes
import pandas as pd
import conceptnet_lite
conceptnet_lite.connect("data/conceptnet.db", db_download_url=None)
from conceptnet_lite import Label, edges_for, edges_between

## Pendientes para el lunes:

- Trabajar en el algoritmo que recupera relaciones de una palabra en un sentido, relaciones de otra palabra en sentido opuesto y checar coincidencias
- Hacer recursivo o iterativo el proceso en las aristas recuperadas si no existe ninguna palabra en común. Poner un límite de revisiones.
- Probar lo mismo pero con los sentidos de búsqueda invertidos para ambas palabras

---
- Juntar la columna de las palabras traducidas e el df de EmoPro para tener acceso a sus socres
- Hacer las divisiones intra e inter emoción discreta (Intra es entre altamente prototípicas e inter es con grupos de otras emociones discretas)

### Información general sobre conceptent_lite

In [3]:
%psource conceptnet_lite

from enum import Enum
from pathlib import Path
from typing import Iterable, Optional, Union

import peewee

from conceptnet_lite.db import CONCEPTNET_EDGE_COUNT, CONCEPTNET_DUMP_DOWNLOAD_URL, CONCEPTNET_DB_NAME
from conceptnet_lite.db import CONCEPTNET_DB_URL
from conceptnet_lite.db import Concept, Language, Label, Relation, RelationName, Edge
from conceptnet_lite.db import prepare_db, _open_db, _generate_db_path, download_db
from conceptnet_lite.utils import PathOrStr, _to_snake_case


def connect(
        db_path: PathOrStr = CONCEPTNET_DB_NAME,
        db_download_url: Optional[str] = CONCEPTNET_DB_URL,
        delete_compressed_db: bool = True,
        dump_download_url: str = CONCEPTNET_DUMP_DOWNLOAD_URL,
        load_dump_edge_count: int = CONCEPTNET_EDGE_COUNT,
        delete_compressed_dump: bool = True,
        delete_dump: bool = True,
) -> None:
    """Connect to ConceptNet database.

    This function connects to ConceptNet database. If it does not exists, there are two opt

In [4]:
dir(conceptnet_lite.CONCEPTNET_DB_NAME)

['__add__',
 '__class__',
 '__contains__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getnewargs__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__mod__',
 '__mul__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__rmod__',
 '__rmul__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 'capitalize',
 'casefold',
 'center',
 'count',
 'encode',
 'endswith',
 'expandtabs',
 'find',
 'format',
 'format_map',
 'index',
 'isalnum',
 'isalpha',
 'isascii',
 'isdecimal',
 'isdigit',
 'isidentifier',
 'islower',
 'isnumeric',
 'isprintable',
 'isspace',
 'istitle',
 'isupper',
 'join',
 'ljust',
 'lower',
 'lstrip',
 'maketrans',
 'partition',
 'removeprefix',
 'removesuffix',
 'replace',
 'rfind',
 'rindex',
 'rjust',
 'rpartition',
 'rsplit',
 'rstrip',
 'split',
 'splitlines',
 'startswith',
 'stri

In [5]:
help(edges_between)

Help on function edges_between in module conceptnet_lite:

edges_between(start_concepts: Iterable[conceptnet_lite.db.Concept], end_concepts: Iterable[conceptnet_lite.db.Concept], relation: Union[conceptnet_lite.db.Relation, str, NoneType] = None, two_way: bool = False) -> peewee.ModelSelect



## Funciones para recuperar aristas del grafo

Las funciones son para recuperar conceptos que une dos conceptos dados, recuperación de aristas en una dirección y recuperación de todas las aristas

In [6]:
# Todas las relaciones entre dos conceptos | es -> español | en -> inglés
def relaciones(wt,wh):
    concepts_wt = Label.get(text=wt, language='es').concepts
    concepts_wh = Label.get(text=wh, language='es').concepts
    for e in edges_between(concepts_wt, concepts_wh,two_way=True):
        print(e.start.text, "-", e.end.text, "|", e.relation.name,e)

In [7]:
# Todas las relaciones entre dos conceptos, con dirección
def relaciones_generales(wt,wh):
    concepts_wt = Label.get(text=wt, language='es').concepts
    concepts_wh = Label.get(text=wh, language='es').concepts
    for e in edges_between(concepts_wt, concepts_wh,two_way=True):
        if wt== e.start.text:
            print(e.start.text, "-", e.end.text, "|", e.relation.name,e)

In [8]:
# Relación con dirección, quitando las relaciones cognitivas
def relacion_direccion(w):
    s1=set()
    try:
        for e in edges_for(Label.get(text=w, language='en').concepts, same_language=True):
            if w== e.start.text:
                print("direccion ->")
                print(e.start.text, "-", e.end.text, "|", e.relation.name,e)
                s1.add(e.end.text)
    except:
        pass
    return s1

In [9]:
# Relación con dirección inversa, quitando las relaciones cognitivas
def relacion_direccion_inversa(w):
    s1=set()
    try:
        for e in edges_for(Label.get(text=w, language='en').concepts, same_language=True):
            if w== e.end.text:
                print("<-direccion")
                print(e.start.text, "-", e.end.text, "|", e.relation.name,e)
                s1.add(e.start.text)
    except:
        pass
    return s1

In [10]:
def relacion_word(w):
    try:
        for e in edges_for(Label.get(text=w, language='en').concepts, same_language=True):
            if w== e.end.text:
                print("derecha")
                print(e.start.text, "-", e.end.text, "|", e.relation.name,e)
            elif w== e.start.text:
                print("izquierda")
                print(e.start.text, "-", e.end.text, "|", e.relation.name,e)
    except:
        pass    

### Ejemplos

In [11]:
gen = relacion_direccion("nervousnesses")

direccion ->
nervousnesses - nervousness | form_of /a/[/r/form_of/,/c/en/nervousnesses/n/,/c/en/nervousness/]


In [12]:
esp = relacion_direccion_inversa("nervousness" )

<-direccion
caffiene - nervousness | causes /a/[/r/causes/,/c/en/caffiene/,/c/en/nervousness/]
<-direccion
going_on_stage - nervousness | causes /a/[/r/causes/,/c/en/going_on_stage/,/c/en/nervousness/]
<-direccion
going_to_school - nervousness | causes /a/[/r/causes/,/c/en/going_to_school/,/c/en/nervousness/]
<-direccion
having_examination - nervousness | causes /a/[/r/causes/,/c/en/having_examination/,/c/en/nervousness/]
<-direccion
putting_on_stand - nervousness | causes /a/[/r/causes/,/c/en/putting_on_stand/,/c/en/nervousness/]
<-direccion
signing_contract - nervousness | causes /a/[/r/causes/,/c/en/signing_contract/,/c/en/nervousness/]
<-direccion
taking_examination - nervousness | causes /a/[/r/causes/,/c/en/taking_examination/,/c/en/nervousness/]
<-direccion
taking_midterm - nervousness | causes /a/[/r/causes/,/c/en/taking_midterm/,/c/en/nervousness/]
<-direccion
taking_oath - nervousness | causes /a/[/r/causes/,/c/en/taking_oath/,/c/en/nervousness/]
<-direccion
nervousnesses - n

In [13]:
esp

{'act_in_play',
 'anxiety',
 'caffiene',
 'checkitis',
 'flop_sweat',
 'going_on_stage',
 'going_to_school',
 "have_butterflies_in_one's_stomach",
 'having_examination',
 "hold_one's_nerve",
 'in_lather',
 'jitter',
 'jitters',
 'nerves',
 'nervosity',
 'nervously',
 'nervousnesses',
 'panic',
 'propose_to_woman',
 'putting_on_stand',
 'restlessness',
 'screaming_abdabs',
 'signing_contract',
 'skittishness',
 'spaz_attack',
 'spine_chiller',
 'spine_chilling',
 'stage_fright',
 'strain',
 'take_exam',
 'taking_examination',
 'taking_midterm',
 'taking_oath',
 'testitis',
 'white_knuckle',
 'willies'}

In [14]:
gen.intersection(esp)

set()

### Importe de palabras de EmoPro para hacer pruebas de búsqueda y recuperación de caminos

In [15]:
# incluir palabras en español y en inglés
with open('palabras_espaniol_ingles_1palabra.csv') as f:
    df = pd.read_csv(f)

In [16]:
df

,Español,English
0,abandonado,abandoned
1,abandonarse,succumb
2,abandono,abandonment
3,abatido,downcast
4,abatimiento,dejection
...,...,...
1281,vital,vital
1282,vitalidad,vitality
1283,vitalizar,vitalize
1284,vulnerabilidad,vulnerability


In [17]:
words = {'esp':[w['Español'] for _,w in df.iterrows()], 'ing': [w['English'] for _,w in df.iterrows()]}

In [18]:
word1 = relacion_direccion(words['ing'][3])

direccion ->
downcast - down | similar_to /a/[/r/similar_to/,/c/en/downcast/a/wn/,/c/en/down/a/wn/]
direccion ->
downcast - down_in_mouth | synonym /a/[/r/synonym/,/c/en/downcast/a/wn/,/c/en/down_in_mouth/a/wn/]
direccion ->
downcast - shaft | is_a /a/[/r/is_a/,/c/en/downcast/n/wn/artifact/,/c/en/shaft/n/wn/artifact/]
direccion ->
downcast - computing | has_context /a/[/r/has_context/,/c/en/downcast/n/,/c/en/computing/]
direccion ->
downcast - mining | has_context /a/[/r/has_context/,/c/en/downcast/n/,/c/en/mining/]
direccion ->
downcast - cast | related_to /a/[/r/related_to/,/c/en/downcast/n/,/c/en/cast/]
direccion ->
downcast - look | related_to /a/[/r/related_to/,/c/en/downcast/n/,/c/en/look/]
direccion ->
downcast - melancholy | related_to /a/[/r/related_to/,/c/en/downcast/n/,/c/en/melancholy/]
direccion ->
downcast - subtype | related_to /a/[/r/related_to/,/c/en/downcast/n/,/c/en/subtype/]
direccion ->
downcast - supertype | related_to /a/[/r/related_to/,/c/en/downcast/n/,/c/en/su

In [19]:
word2 = relacion_direccion_inversa(words['ing'][45])

<-direccion
eagernesses - eagerness | form_of /a/[/r/form_of/,/c/en/eagernesses/n/,/c/en/eagerness/]
<-direccion
alacrity - eagerness | related_to /a/[/r/related_to/,/c/en/alacrity/n/,/c/en/eagerness/]
<-direccion
anticipation - eagerness | related_to /a/[/r/related_to/,/c/en/anticipation/n/,/c/en/eagerness/]
<-direccion
ardency - eagerness | related_to /a/[/r/related_to/,/c/en/ardency/n/,/c/en/eagerness/]
<-direccion
avidity - eagerness | related_to /a/[/r/related_to/,/c/en/avidity/n/,/c/en/eagerness/]
<-direccion
bloodthirst - eagerness | related_to /a/[/r/related_to/,/c/en/bloodthirst/n/,/c/en/eagerness/]
<-direccion
eagerly - eagerness | related_to /a/[/r/related_to/,/c/en/eagerly/r/,/c/en/eagerness/]
<-direccion
enthusiasm - eagerness | related_to /a/[/r/related_to/,/c/en/enthusiasm/n/,/c/en/eagerness/]
<-direccion
excitement - eagerness | related_to /a/[/r/related_to/,/c/en/excitement/,/c/en/eagerness/]
<-direccion
impatience - eagerness | related_to /a/[/r/related_to/,/c/en/impa

In [20]:
inter = word1.intersection(word2)

In [24]:
path = set()
for w in word1:
    for c in word2:
        relations = relacion_direccion(w)
        antirelation = relacion_direccion_inversa(c)
        path = relations.intersection(antirelation)
        print(path)
        if len(path) != 0:
            print(path)
        break
if len(path) == 0:
    print('No coincidencia')



direccion ->
melancholy - mirthful | antonym /a/[/r/antonym/,/c/en/melancholy/,/c/en/mirthful/]
direccion ->
melancholy - emotion | is_a /a/[/r/is_a/,/c/en/melancholy/,/c/en/emotion/]
direccion ->
melancholy - deep | related_to /a/[/r/related_to/,/c/en/melancholy/,/c/en/deep/]
direccion ->
melancholy - deep_sadness | related_to /a/[/r/related_to/,/c/en/melancholy/,/c/en/deep_sadness/]
direccion ->
melancholy - depressed | related_to /a/[/r/related_to/,/c/en/melancholy/,/c/en/depressed/]
direccion ->
melancholy - depressed_feeling | related_to /a/[/r/related_to/,/c/en/melancholy/,/c/en/depressed_feeling/]
direccion ->
melancholy - depression | related_to /a/[/r/related_to/,/c/en/melancholy/,/c/en/depression/]
direccion ->
melancholy - describer | related_to /a/[/r/related_to/,/c/en/melancholy/,/c/en/describer/]
direccion ->
melancholy - feeling | related_to /a/[/r/related_to/,/c/en/melancholy/,/c/en/feeling/]
direccion ->
melancholy - infinite | related_to /a/[/r/related_to/,/c/en/melan

In [ ]:
relaciones1 = relacion_direccion(df.at[0, 'English'])

direccion ->
abandoned - abandon | derived_from /a/[/r/derived_from/,/c/en/abandoned/,/c/en/abandon/v/]
direccion ->
abandoned - abandon | derived_from /a/[/r/derived_from/,/c/en/abandoned/,/c/en/abandon/v/wikt/en_1/]
direccion ->
abandoned - abandon | etymologically_related_to /a/[/r/etymologically_related_to/,/c/en/abandoned/,/c/en/abandon/]
direccion ->
abandoned - abandon | form_of /a/[/r/form_of/,/c/en/abandoned/,/c/en/abandon/v/]
direccion ->
abandoned - recovered | distinct_from /a/[/r/distinct_from/,/c/en/abandoned/a/,/c/en/recovered/]
direccion ->
abandoned - restored | distinct_from /a/[/r/distinct_from/,/c/en/abandoned/a/,/c/en/restored/]
direccion ->
abandoned - saved | distinct_from /a/[/r/distinct_from/,/c/en/abandoned/a/,/c/en/saved/]
direccion ->
abandoned - geology | has_context /a/[/r/has_context/,/c/en/abandoned/a/,/c/en/geology/]
direccion ->
abandoned - deserted | related_to /a/[/r/related_to/,/c/en/abandoned/a/,/c/en/deserted/]
direccion ->
abandoned - extremely |

{'abandon',
 'careless',
 'cast_aside',
 'cast_away',
 'cast_off',
 'corrupt',
 'corrupted',
 'demitted',
 'demoralized',
 'depraved',
 'derelict',
 'deserted',
 'discarded',
 'dissolute',
 'extremely',
 'forsaken',
 'geology',
 'given_over',
 'given_up',
 'graceless',
 'hardened',
 'immoral',
 'impenitent',
 'impetuous',
 'incorrigible',
 'irreclaimable',
 'irreclaimably',
 'obdurate',
 'outcast',
 'profligate',
 'reckless',
 'recovered',
 'rejected',
 'relinquished',
 'reprobate',
 'restored',
 'saved',
 'shameless',
 'sinning',
 'uninhabited',
 'uninhibited',
 'unprincipled',
 'unrestrained',
 'vice',
 'vicious',
 'vile',
 'wanton',
 'wicked',
 'wild'}

In [26]:
relaciones2=relacion_direccion_inversa(df.at[2, 'English'])

<-direccion
abandonments - abandonment | form_of /a/[/r/form_of/,/c/en/abandonments/,/c/en/abandonment/n/]
<-direccion
acquisition - abandonment | antonym /a/[/r/antonym/,/c/en/acquisition/n/,/c/en/abandonment/]
<-direccion
arrogation - abandonment | antonym /a/[/r/antonym/,/c/en/arrogation/n/,/c/en/abandonment/]
<-direccion
abandonments - abandonment | form_of /a/[/r/form_of/,/c/en/abandonments/n/,/c/en/abandonment/]
<-direccion
abandon - abandonment | related_to /a/[/r/related_to/,/c/en/abandon/n/wikt/en_2/,/c/en/abandonment/]
<-direccion
abandon - abandonment | related_to /a/[/r/related_to/,/c/en/abandon/v/wikt/en_1/,/c/en/abandonment/]
<-direccion
abandonable - abandonment | related_to /a/[/r/related_to/,/c/en/abandonable/a/,/c/en/abandonment/]
<-direccion
abandoning - abandonment | related_to /a/[/r/related_to/,/c/en/abandoning/n/,/c/en/abandonment/]
<-direccion
abandonments - abandonment | related_to /a/[/r/related_to/,/c/en/abandonments/n/,/c/en/abandonment/]
<-direccion
abandon

In [27]:
union = relaciones1.intersection(relaciones2)

In [28]:
union

{'abandon'}

In [16]:
relacion_word(df.at[0, 'English'])

derecha
abandannaad - abandoned | derived_from /a/[/r/derived_from/,/c/en/abandannaad/,/c/en/abandoned/]
derecha
abandonedly - abandoned | derived_from /a/[/r/derived_from/,/c/en/abandonedly/,/c/en/abandoned/]
derecha
abandonedness - abandoned | derived_from /a/[/r/derived_from/,/c/en/abandonedness/,/c/en/abandoned/]
derecha
nonabandoned - abandoned | derived_from /a/[/r/derived_from/,/c/en/nonabandoned/,/c/en/abandoned/]
derecha
semiabandoned - abandoned | derived_from /a/[/r/derived_from/,/c/en/semiabandoned/,/c/en/abandoned/]
derecha
unabandoned - abandoned | derived_from /a/[/r/derived_from/,/c/en/unabandoned/,/c/en/abandoned/]
derecha
abandoned_habits - abandoned | etymologically_related_to /a/[/r/etymologically_related_to/,/c/en/abandoned_habits/,/c/en/abandoned/]
derecha
dog - abandoned | not_desires /a/[/r/not_desires/,/c/en/dog/,/c/en/abandoned/]
derecha
person - abandoned | not_desires /a/[/r/not_desires/,/c/en/person/,/c/en/abandoned/]
derecha
abandonable - abandoned | relat

In [18]:
relacion_word(df.at[0, 'Español'])

izquierda
abandonado - abandonar | etymologically_related_to /a/[/r/etymologically_related_to/,/c/es/abandonado/,/c/es/abandonar/]
izquierda
abandonado - abandonar | form_of /a/[/r/form_of/,/c/es/abandonado/,/c/es/abandonar/v/]
izquierda
abandonado - abandonar | form_of /a/[/r/form_of/,/c/es/abandonado/v/,/c/es/abandonar/]
izquierda
abandonado - abandonar | related_to /a/[/r/related_to/,/c/es/abandonado/v/,/c/es/abandonar/]


In [53]:
relacion_word("nervioso")

derecha
nervio - nervioso | related_to /a/[/r/related_to/,/c/es/nervio/n/,/c/es/nervioso/]
derecha
nervosidad - nervioso | related_to /a/[/r/related_to/,/c/es/nervosidad/n/,/c/es/nervioso/]


In [55]:
relacion_word("nervosidad")

derecha
nervio - nervosidad | related_to /a/[/r/related_to/,/c/es/nervio/n/,/c/es/nervosidad/]
izquierda
nervosidad - nervio | related_to /a/[/r/related_to/,/c/es/nervosidad/n/,/c/es/nervio/]
izquierda
nervosidad - nerviosismo | related_to /a/[/r/related_to/,/c/es/nervosidad/n/,/c/es/nerviosismo/]
izquierda
nervosidad - nervioso | related_to /a/[/r/related_to/,/c/es/nervosidad/n/,/c/es/nervioso/]
izquierda
nervosidad - nerviosismo | synonym /a/[/r/synonym/,/c/es/nervosidad/n/,/c/es/nerviosismo/]


# relaciones entre dos conceptos

In [11]:
relaciones("try","look")

look - try | related_to /a/[/r/related_to/,/c/en/look/v/,/c/en/try/]


In [12]:
relaciones("man","car")

In [13]:
relaciones("side","full")

In [45]:
relaciones("red","white")

In [43]:
relacion_word("state")

derecha
city - state | antonym /a/[/r/antonym/,/c/en/city/,/c/en/state/]
derecha
colony - state | antonym /a/[/r/antonym/,/c/en/colony/,/c/en/state/]
derecha
country - state | antonym /a/[/r/antonym/,/c/en/country/,/c/en/state/]
derecha
nation - state | antonym /a/[/r/antonym/,/c/en/nation/,/c/en/state/]
derecha
capital - state | at_location /a/[/r/at_location/,/c/en/capital/,/c/en/state/]
derecha
city - state | at_location /a/[/r/at_location/,/c/en/city/,/c/en/state/]
derecha
county - state | at_location /a/[/r/at_location/,/c/en/county/,/c/en/state/]
derecha
highway - state | at_location /a/[/r/at_location/,/c/en/highway/,/c/en/state/]
derecha
human - state | at_location /a/[/r/at_location/,/c/en/human/,/c/en/state/]
derecha
interstate_highway - state | at_location /a/[/r/at_location/,/c/en/interstate_highway/,/c/en/state/]
derecha
national_interstate_highway - state | at_location /a/[/r/at_location/,/c/en/national_interstate_highway/,/c/en/state/]
derecha
state_highway - state | at_

In [44]:
relacion_word("american")

derecha
foreign - american | antonym /a/[/r/antonym/,/c/en/foreign/,/c/en/american/]
derecha
afrimerican - american | derived_from /a/[/r/derived_from/,/c/en/afrimerican/,/c/en/american/]
derecha
afro_american - american | derived_from /a/[/r/derived_from/,/c/en/afro_american/,/c/en/american/]
derecha
afromerican - american | derived_from /a/[/r/derived_from/,/c/en/afromerican/,/c/en/american/]
derecha
amerasian - american | derived_from /a/[/r/derived_from/,/c/en/amerasian/,/c/en/american/]
derecha
americanese - american | derived_from /a/[/r/derived_from/,/c/en/americanese/,/c/en/american/]
derecha
americanesque - american | derived_from /a/[/r/derived_from/,/c/en/americanesque/,/c/en/american/]
derecha
americaness - american | derived_from /a/[/r/derived_from/,/c/en/americaness/,/c/en/american/]
derecha
americanian - american | derived_from /a/[/r/derived_from/,/c/en/americanian/,/c/en/american/]
derecha
americanime - american | derived_from /a/[/r/derived_from/,/c/en/americanime/,/

# Diccionarios de relaciones

In [2]:
#cargar relaciones para trabajar de manera local
df_diccionario = pd.read_pickle("data/Relaciones_generales.pickle")
df_diccionario_generales = df_diccionario.to_dict()

df_diccionario = pd.read_pickle("data/Relaciones_especificas.pickle")
df_diccionario_especificas = df_diccionario.to_dict()

In [9]:
"sdasd" in df_diccionario_generales

False

In [10]:
df_diccionario_generales["sdasd"]["is_a"]

KeyError: 'sdasd'

In [8]:
df_diccionario_especificas["man"]["is_a"]

{'adonis',
 'babu',
 'bachelor',
 'beatles',
 'bey',
 'black',
 'bolivian_businessman',
 'bouncer',
 'boy',
 'boyfriend',
 'british_serviceman',
 'bull',
 'catholic_priest',
 'checker',
 'chessman',
 'colombian_man',
 'crewman',
 'dandy',
 'ejaculator',
 'esq',
 'eunuch',
 'ex_boyfriend',
 'ex_husband',
 'father',
 'father_figure',
 'fellow',
 'fireman',
 'first_gentleman',
 'french_man',
 'galoot',
 'geezer',
 'gentleman',
 'godfather',
 'grass_widower',
 'guy',
 'herr',
 'hooray_henry',
 'housefather',
 'hunk',
 'inamorato',
 'iron_man',
 'ironside',
 'israeli_border_policeman',
 'junkman',
 'man_armed_with_knives',
 'mens_size_10_shoe_size',
 'middle_aged_man',
 'milk_man',
 'monsieur',
 'old_boy',
 'old_man',
 'patriarch',
 'peter_pan',
 'policeman',
 'ponce',
 'posseman',
 'senhor',
 'shaver',
 'signor',
 'signore',
 'sir',
 'stiff',
 'stud',
 'tall_man',
 'tarzan',
 'tile',
 'white',
 'white_man',
 'widower',
 'woman',
 'womanizer',
 'wonder_boy',
 'young_buck'}